# El modelo de **Izhikevich**

## **Ejercicio 1)** Importando librerías

**1)** Importe las librerías `numpy` para operar con arrays, `scipy` para utilizar rutinas de algebra lineal y `matplotlib.pyplot` para graficar.

**2)** Incorpore a la notebook el integrador de ODEs de la guía anterior.

In [ ]:
# 1.1) Importación de librerías científicas
import numpy as np
import scipy as sp
import scipy.linalg as linalg
import matplotlib.pyplot as plt

# Configuración estética para figuras y gráficos
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
# 1.2) Integradores de ODEs: Euler, Runge-Kutta 4 e integrador con función condicionante

def euler(f, x, t, h, p):
    """
    Calcula un paso de integración del método de Euler.

    Argumentos de entrada:
        f : R^n, R, p -> R^n
        x = x(t) : R^n
        t = tiempo : R
        h = paso de tiempo : R
        p = parametros : R^q

    Retorna aproximacion numérica de:
        x(t+h) : R^n
    según el método de Euler.
    """
    return x + h * f(x, t, p)


def rk4(f, x, t, h, p):
    """
    Calcula un paso de integración del método de Runge Kutta orden 4.

    Argumentos de entrada:
        f : R^n, R, p -> R^n
        x = x(t) : R^n
        t = tiempo : R
        h = paso de tiempo : R
        p = parametros : R^q

    Retorna aproximacion numérica de:
        x(t+h) : R^n
    según el método de Runge-Kutta de orden 4.
    """
    k1 = f(x, t, p)
    k2 = f(x + 0.5 * h * k1, t + 0.5 * h, p)
    k3 = f(x + 0.5 * h * k2, t + 0.5 * h, p)
    k4 = f(x + h * k3, t + h, p)
    return x + h * (k1 + 2.0 * k2 + 2.0 * k3 + k4) / 6.0


def integrador_ode(m, f, xa, a, b, k, p, c=lambda x, t, p: x):
    """
    Integra numéricamente la ODE
        dx/dt = f(x,t)
    sobre el intervalo t:[a,b] usando k pasos de integración y el método m, bajo condicion inicial x(a)=x0.
    No es necesario que a<b.

    Argumentos de entrada:
        m = metodo de integracion (ej. euler, rk4, etc.)
        f : R^n, R, p -> R^n
        xa = condicion inicial : R^n
        a = tiempo inicial : R
        b = tiempo final : R
        k = num. pasos de integracion : N
        p = parametros : R^q
        c = función condicionante : R^n,R,p -> R^n

    Retorna:
        t : R^{k+1} , t_j = a+j*h para j=0,1,...,k
        w : R^{n,k+1} , w_ij = x_i(t_j) para i=0,1,...,n-1 y j=0,1,...,k
    """
    assert k > 0
    n = len(xa)
    h = (b - a) / k
    w = np.zeros((n, k + 1))
    t = np.zeros(k + 1)
    w[:, 0] = xa
    t[0] = a

    for j in range(k):
        t[j + 1] = t[j] + h
        w[:, j + 1] = m(f, w[:, j], t[j], h, p)
        w[:, j + 1] = c(w[:, j + 1], t[j + 1], p)

    return t, w

## **Ejercicio 2)** Modelo de Izhikevich

El modelo de Izhikevich (ref. 1.) es una simplificación del conocido modelo que Hodgkin y Huxley introdujeron en 1952 (ref. 2. y premio Nobel en Fisiología y Medicina de 1963) para describir el comportamiento del potencial de disparo de una neurona.

El modelo de Izhikevich se describe en términos de la siguiente ODE bidimensional y de primer orden

\begin{eqnarray}
v'(t) &=& g_2v^2(t) + g_1v(t) + g_0 - u(t) + I(t) \\
u'(t) &=& b(cv(t)-u(t))
\end{eqnarray}

unida al mecanismo de *reseteo* del potencial

\begin{eqnarray}
v(t) &\leftarrow& v_- \\
u(t) &\leftarrow& u(t) + \Delta u \\
\end{eqnarray}

la cuál emula el disparo de la neurona, imponiendose cuando la condición $v(t)\geq v_+$ se cumple.

En todas estas ecuaciones, $g_2$, $g_1$, $g_0$, $v_-$, $v_+$, $b$, $c$ y $\Delta u$ son parámetros del modelo.

Además, se considera la función dependiente del tiempo $I:\mathbb{R}\ni t\to I(t)\in \mathbb{R}$, la cual representa una corriente de entrada que exita a la neurona.

Referencias:

1. [Eugene M. Izhikevich, Simple Model of Spiking Neurons, IEEE TRANSACTIONS ON NEURAL NETWORKS, VOL. 14, NO. 6, P. 1569, NOVEMBER 2003](https://ieeexplore.ieee.org/document/1257420)
  
    [PDF de acceso libre](https://www.izhikevich.org/publications/spikes.pdf)
2. https://en.wikipedia.org/wiki/Hodgkin%E2%80%93Huxley_model

***

**1)** Integre el modelo de Izhikevich usando el método RK4 en el intérvalo de tiempo

* $t\in [0,200]$

Considere, para ello, la siguiente elección de valores de los parámetros (pero pruebe variaciones para los distintos casos)

* $g_2=0.04$

* $g_1=5$

* $g_0=140$

* $v_-=-65$

* $v_+=30$

* $b=0.02$

* $c=0.2$

* $\Delta u=2$

una corriente de entrada

\begin{eqnarray}
I(t)
&=&
\left\{
\begin{array}{ll}
0  & t < 10 \\
10 & c.c.
\end{array}
\right.
\end{eqnarray}

y comience probando con la condición inicial

* $v(0) = -70$

* $u(0) = cv(0)$

Además, utilice el paso de integración

$h=0.1$

**2)** Grafique la solución.

**3)** Reproduzca, aproximadamente, los últimos 8 paneles de la figura 2 del paper en la referencia 1.

Para ello, tenga en cuenta la siguiente tabla confeccionada a partir de la información en la figura 2 del paper

$$
\begin{aligned}
& \text {Tabla 1. Parmaterización por caso. }\\
&\begin{array}{ccccc}
\hline \hline
\text { Caso } & b & c & v_- & \Delta u & I \\
\hline
\text { RS }  & 0.02 & 0.2  & -65 & 8    & I_1 \\
\text { IB }  & 0.02 & 0.2  & -55 & 4    & I_1 \\
\text { CH }  & 0.02 & 0.2  & -50 & 2    & I_1 \\
\text { FS }  & 0.1  & 0.2  & -65 & 2    & I_1 \\
\text { TC1 } & 0.02 & 0.25 & -65 & 0.05 & I_2 \\
\text { TC2 } & 0.02 & 0.25 & -65 & 0.05 & I_3 \\
\text { RZ }  & 0.1  & 0.26 & -65 & 2    & I_4 \\
\text { LTS } & 0.02 & 0.25 & -65 & 2    & I_1 \\
\hline
\end{array}
\end{aligned}
$$

donde

\begin{eqnarray}
I_1(t)
&=&
\left\{
\begin{array}{ll}
0  & t < 25 \\
10 & c.c.
\end{array}
\right. \\
I_2(t)
&=&
\left\{
\begin{array}{ll}
0  & t < 150 \\
2.5 & c.c.
\end{array}
\right. \\
I_3(t)
&=&
\left\{
\begin{array}{ll}
-15  & t < 100 \\
0 & c.c.
\end{array}
\right. \\
I_4(t)
&=&
\left\{
\begin{array}{ll}
0    & t < 10 \\
.2 & 10 \leq t < 100 \\
10   & 100 \leq t < 101 \\
.2 & c.c.
\end{array}
\right.
\end{eqnarray}

In [ ]:
# 2.1) Integración del modelo de Izhikevich usando RK4 con reseteo externo al integrador

def f_izhikevich(x, t, p):
    """
    Campo vectorial del sistema de Izhikevich continuo (sin mecanismo de reseteo):
        v'(t) = g2*v^2(t) + g1*v(t) + g0 - u(t) + I(t)
        u'(t) = b*(c*v(t) - u(t))
    
    Argumentos:
        x : array de estado [v, u]
        t : tiempo actual
        p : diccionario con parámetros (g2, g1, g0, b, c, I)
    """
    v, u = x[0], x[1]
    # I(t) puede ser función del tiempo o constante escalar
    I_val = p['I'](t) if callable(p['I']) else p['I']
    
    dv = p['g2'] * (v**2) + p['g1'] * v + p['g0'] - u + I_val
    du = p['b'] * (p['c'] * v - u)
    return np.array([dv, du], dtype=float)


def condicion_reseteo_izhikevich(x, t, p):
    """
    Condición de reseteo discontinuo para integrador_ode:
    Si v >= v_+: v <- v_-, u <- u + Delta_u
    """
    x_res = np.array(x, dtype=float)
    if x_res[0] >= p['v_plus']:
        x_res[0] = p['v_minus']
        x_res[1] += p['delta_u']
    return x_res


def simular_izhikevich_rk4(p, I_func, t_span=(0.0, 200.0), h=0.1, x0=None):
    """
    Integra el modelo de Izhikevich con Runge-Kutta 4 (RK4).
    IMPORTANTE: El reseteo v <- v_-, u <- u + Delta_u se implementa estrictamente FUERA
    del integrador RK4, chequeando después de cada paso si v >= v_+.
    
    Para permitir una graficación realista del pico del potencial de acción (+30 mV) como en el paper,
    se almacena v_plus en el instante del spike antes de resetear las variables dinámicas de integración.
    
    Retorna:
        t      : array temporal de dimensión (k+1,)
        w      : array de estados dinámicos [v, u] de dimensión (2, k+1)
        v_plot : array con los valores de v incluyendo los picos de espiga a +30 mV
    """
    t_start, t_end = t_span
    k = int(np.round((t_end - t_start) / h))
    t = np.linspace(t_start, t_end, k + 1)
    
    params = p.copy()
    params['I'] = I_func
    
    if x0 is None:
        v0 = -70.0
        u0 = params['c'] * v0
        x0 = np.array([v0, u0], dtype=float)
        
    w = np.zeros((2, k + 1))
    v_plot = np.zeros(k + 1)
    
    w[:, 0] = x0
    v_plot[0] = x0[0]
    
    for j in range(k):
        # 1. Paso de integración continua RK4 sin alterar x en las etapas intermedias
        x_next = rk4(f_izhikevich, w[:, j], t[j], h, params)
        
        # 2. Mecanismo de disparo y reseteo evaluado FUERA del integrador
        if x_next[0] >= params['v_plus']:
            # Guardamos el valor de pico (+30 mV) para el trazado gráfico
            v_plot[j + 1] = params['v_plus']
            # Reseteo discontinuo de las variables dinámicas para el siguiente paso
            w[0, j + 1] = params['v_minus']
            w[1, j + 1] = x_next[1] + params['delta_u']
        else:
            v_plot[j + 1] = x_next[0]
            w[:, j + 1] = x_next
            
    return t, w, v_plot


# Definición de parámetros del Caso Base
params_base = {
    'g2': 0.04,
    'g1': 5.0,
    'g0': 140.0,
    'v_minus': -65.0,
    'v_plus': 30.0,
    'b': 0.02,
    'c': 0.2,
    'delta_u': 2.0
}

# Corriente de entrada base: I(t) = 0 si t < 10, 10 en caso contrario
def I_base(t):
    return 0.0 if t < 10.0 else 10.0

# Paso de integración y rango temporal
h = 0.1
t_span = (0.0, 200.0)
x0_base = np.array([-70.0, params_base['c'] * (-70.0)]) # v(0) = -70, u(0) = c*v(0) = -14

# Integración del caso base con RK4
t_base, w_base, v_plot_base = simular_izhikevich_rk4(params_base, I_base, t_span=t_span, h=h, x0=x0_base)

# Verificación con integrador_ode provisto en el Ejercicio 1.2 usando la función condicionante c
params_base_ode = params_base.copy()
params_base_ode['I'] = I_base
k_base = int(np.round((t_span[1] - t_span[0]) / h))
t_ode, w_ode = integrador_ode(rk4, f_izhikevich, x0_base, t_span[0], t_span[1], k_base, params_base_ode, c=condicion_reseteo_izhikevich)

print("Integración del caso base completada exitosamente:")
print(f"- Intervalo de tiempo: [{t_base[0]:.1f}, {t_base[-1]:.1f}] ms, paso h = {h} ms")
print(f"- Cantidad total de pasos: {len(t_base)}")
print(f"- Número de disparos registrados: {np.sum(v_plot_base >= params_base['v_plus'])}")

In [ ]:
# 2.2) Graficación de las soluciones v(t) vs t y u(t) vs t para el caso base

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

# Panel Superior: Potencial de membrana v(t)
ax1.plot(t_base, v_plot_base, color='navy', lw=1.2, label='$v(t)$ (Potencial de membrana)')
ax1.axhline(params_base['v_plus'], color='crimson', ls='--', lw=1, alpha=0.7, label='$v_+ = 30$ mV (Umbral de disparo)')
ax1.axhline(params_base['v_minus'], color='forestgreen', ls=':', lw=1, alpha=0.7, label='$v_- = -65$ mV (Reseteo)')
ax1.set_ylabel('$v$ [mV]', fontsize=12)
ax1.set_title('Modelo de Izhikevich - Caso Base ($t \in [0, 200]$, $h = 0.1$ ms)', fontsize=13, fontweight='bold')
ax1.set_ylim(-85, 40)
ax1.legend(loc='upper right', framealpha=0.9)
ax1.grid(True, alpha=0.3)

# Panel Inferior: Variable de recuperación u(t) y corriente I(t)
ax2.plot(t_base, w_base[1, :], color='darkorange', lw=1.5, label='$u(t)$ (Variable de recuperación)')
ax2.set_xlabel('Tiempo $t$ [ms]', fontsize=12)
ax2.set_ylabel('$u$ [pA]', fontsize=12)

# Eje secundario para visualizar la corriente escalón inyectada I(t)
ax2_twin = ax2.twinx()
I_base_vals = np.array([I_base(ti) for ti in t_base])
ax2_twin.plot(t_base, I_base_vals, color='gray', ls='--', lw=1.2, label='$I(t)$ (Corriente)')
ax2_twin.set_ylabel('$I$ [pA]', color='gray', fontsize=11)
ax2_twin.tick_params(axis='y', labelcolor='gray')
ax2_twin.set_ylim(-2, 16)

# Unificar leyendas
handles_ax2, labels_ax2 = ax2.get_legend_handles_labels()
handles_twin, labels_twin = ax2_twin.get_legend_handles_labels()
ax2.legend(handles_ax2 + handles_twin, labels_ax2 + labels_twin, loc='upper left', framealpha=0.9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### **Discusión teórica: Los 8 patrones electrofisiológicos de la Figura 2**

El modelo propuesto por Eugene M. Izhikevich logra capturar una amplia variedad de comportamientos dinámicos neuronales mediante la variación de apenas 4 parámetros clave ($b, c, v_-, \Delta u$ en la notación de la guía; correspondientes a $a, b, c, d$ en el paper original):

1. **RS (Regular Spiking):** Típica neurona piramidal excitatoria cortical. Al recibir un estímulo escalón sostenido, dispara inicialmente con una frecuencia elevada y luego experimenta una marcada *adaptación de frecuencia de disparo* (spike-frequency adaptation), estabilizándose en una frecuencia regular menor debido a la acumulación lenta de la variable de recuperación $u$.
2. **IB (Intrinsically Bursting):** Dispara una ráfaga inicial densa de potenciales de acción (*burst*), seguida de disparos tónicos regulares espaciados. Es característica de neuronas en las capas profundas de la corteza cerebral.
3. **CH (Chattering):** Exhibe trenes periódicos de ráfagas de alta frecuencia separadas por breves intervalos de reposo. Se asocia con neuronas oscilatorias corticales que modulan el ritmo gamma.
4. **FS (Fast Spiking):** Interneuronas inhibitorias corticales que disparan a frecuencias sumamente elevadas (varios cientos de Hz) sin exhibir prácticamente adaptación de frecuencia de disparo, debido a un rápido restablecimiento de $u$ ($b=0.1$).
5. **TC1 (Thalamo-Cortical Tonic):** Neurona tálamo-cortical que en condiciones basales despolarizantes ($I_2 > 0$) dispara potenciales tónicos regulares continuos.
6. **TC2 (Thalamo-Cortical Rebound Burst):** Neurona tálamo-cortical sometida a una corriente hiperpolarizante negativa ($I_3 = -15$). Al cesar bruscamente la inhibición en $t=100$ ms (*rebound* o rebote post-inhibitorio), se dispara una ráfaga transitoria debida a la desinactivación de corrientes de bajo umbral.
7. **RZ (Resonator):** Manifiesta resonancia subumbral ante frecuencias preferenciales; una corriente pequeña produce oscilaciones amortiguadas, pero un pulso breve e intenso ($100 \le t < 101$ ms) desencadena un disparo resonante.
8. **LTS (Low-Threshold Spiking):** Interneuronas inhibitorias de bajo umbral que responden con un disparo ráfaga inicial y luego mantienen una adaptación progresiva con frecuencia basal baja.

In [ ]:
# 2.3) Reproducción de los 8 paneles de la Figura 2 del paper de Izhikevich

# Definición de las corrientes de entrada especificadas en la Tabla 1
def I1(t):
    return 0.0 if t < 25.0 else 10.0

def I2(t):
    return 0.0 if t < 150.0 else 2.5

def I3(t):
    return -15.0 if t < 100.0 else 0.0

def I4(t):
    if t < 10.0:
        return 0.0
    elif 10.0 <= t < 100.0:
        return 0.2
    elif 100.0 <= t < 101.0:
        return 10.0
    else:
        return 0.2

# Diccionario con la parametrización de los 8 casos según la Tabla 1 del notebook
casos_fig2 = {
    'RS (Regular Spiking)': {
        'b': 0.02, 'c': 0.2, 'v_minus': -65.0, 'delta_u': 8.0,
        'I_func': I1, 't_span': (0.0, 200.0), 'v0': -70.0
    },
    'IB (Intrinsically Bursting)': {
        'b': 0.02, 'c': 0.2, 'v_minus': -55.0, 'delta_u': 4.0,
        'I_func': I1, 't_span': (0.0, 200.0), 'v0': -70.0
    },
    'CH (Chattering)': {
        'b': 0.02, 'c': 0.2, 'v_minus': -50.0, 'delta_u': 2.0,
        'I_func': I1, 't_span': (0.0, 200.0), 'v0': -70.0
    },
    'FS (Fast Spiking)': {
        'b': 0.1,  'c': 0.2, 'v_minus': -65.0, 'delta_u': 2.0,
        'I_func': I1, 't_span': (0.0, 200.0), 'v0': -70.0
    },
    'TC1 (Thalamo-Cortical Tonic)': {
        'b': 0.02, 'c': 0.25, 'v_minus': -65.0, 'delta_u': 0.05,
        'I_func': I2, 't_span': (0.0, 250.0), 'v0': -70.0
    },
    'TC2 (Thalamo-Cortical Rebound)': {
        'b': 0.02, 'c': 0.25, 'v_minus': -65.0, 'delta_u': 0.05,
        'I_func': I3, 't_span': (0.0, 200.0), 'v0': -70.0
    },
    'RZ (Resonator)': {
        'b': 0.1,  'c': 0.26, 'v_minus': -65.0, 'delta_u': 2.0,
        'I_func': I4, 't_span': (0.0, 200.0), 'v0': -70.0
    },
    'LTS (Low-Threshold Spiking)': {
        'b': 0.02, 'c': 0.25, 'v_minus': -65.0, 'delta_u': 2.0,
        'I_func': I1, 't_span': (0.0, 200.0), 'v0': -70.0
    }
}

# Configuración del layout: 4 filas x 2 columnas
fig, axes = plt.subplots(4, 2, figsize=(14, 13))
axes = axes.flatten()

for idx, (nombre, cfg) in enumerate(casos_fig2.items()):
    ax = axes[idx]
    
    # Parámetros del caso
    p = {
        'g2': 0.04, 'g1': 5.0, 'g0': 140.0,
        'b': cfg['b'], 'c': cfg['c'],
        'v_minus': cfg['v_minus'], 'v_plus': 30.0,
        'delta_u': cfg['delta_u']
    }
    
    v0 = cfg['v0']
    u0 = cfg['c'] * v0
    x0 = np.array([v0, u0], dtype=float)
    
    # Simulación con RK4 (h = 0.1 ms)
    t_c, w_c, v_plot_c = simular_izhikevich_rk4(p, cfg['I_func'], t_span=cfg['t_span'], h=0.1, x0=x0)
    
    # Curva del potencial de membrana v(t)
    ax.plot(t_c, v_plot_c, color='navy', lw=1.1, label='$v(t)$')
    
    # Eje secundario para la corriente de inyección I(t)
    ax_twin = ax.twinx()
    I_profile = np.array([cfg['I_func'](ti) for ti in t_c])
    ax_twin.plot(t_c, I_profile, color='crimson', ls='-', lw=1.1, alpha=0.55, label='$I(t)$')
    ax_twin.set_ylabel('$I$ [pA]', color='crimson', fontsize=9)
    ax_twin.tick_params(axis='y', labelcolor='crimson', labelsize=8)
    
    # Escala vertical de corriente ajustada para no superponerse con el voltaje
    I_min, I_max = np.min(I_profile), np.max(I_profile)
    margin_I = max(1.0, (I_max - I_min) * 0.35)
    ax_twin.set_ylim(I_min - margin_I, I_max + margin_I * 3.5)
    
    ax.set_title(nombre, fontsize=11, fontweight='bold', pad=4)
    ax.set_ylabel('$v$ [mV]', fontsize=10)
    ax.set_ylim(-90, 42)
    ax.set_xlim(cfg['t_span'][0], cfg['t_span'][1])
    ax.grid(True, alpha=0.25)
    
    if idx >= 6:
        ax.set_xlabel('Tiempo $t$ [ms]', fontsize=10)

plt.suptitle('Figura 2: Reproducción de los 8 patrones electrofisiológicos del paper de Izhikevich', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.subplots_adjust(top=0.95, hspace=0.35, wspace=0.28)
plt.show()

## **Ejercicio 3)** Red de neuronas de Izhikevich

### El método de Euler-Maruyama

Nos interesa resolver una Ecuación Diferencial Estocástica (SDE) de la forma

$$dX_t = f(X_t,t)dt + g(X_t,t)dW_t\;\;\;\; (1)$$

donde, para cada $t\in \mathbb{R}$, las variables $X_t,W_t$ son estocásticas con soporte en $\mathbb{R}$, y $f,g \in (\mathbb{R}^2\to \mathbb{R})$ son funciones relativamente arbitrarias.
Como función de $t$, asuminos que $W_t$ describe un proceso de Wiener. Es decir, asumimos que

1. $W_t$ es una variable estocástica normalmente distribuida de valor esperado $\langle W_t \rangle = 0$ y varianza $\langle (W_t-\langle W_t\rangle)^2 \rangle = \langle W_t^2 \rangle = wt$ para algún $w\geq 0$, y

2. las variables estocásticas $W_{t+s}-W_s$ (caminatas futuras) y $W_s$ (caminatas pasadas) son estadísticamente independientes para todo $t\geq 0$ y $s\leq t$.

Nos interesa resolver la Ec. (1) en un intervalo $t\in [0,T]$ para una condición inicial $X_0=x_0$.

El método de Euler-Maruyama provee una aproximación $Y_t \approx X_t$ para $t\in \{0,\tau,2\tau,...,T\}$ donde $\tau=T/N$ para algún $N\gg 1$ fijo, y viene dado por la cadena de Markov

$$Y_{t+\tau} = Y_t + f(Y_t,t)\tau + g(Y_t,t)w\sqrt{\tau}\mathcal{N}\;\;\;\; (2)$$

donde $Y_0=x_0$ y $\mathcal{N}$ es una variable aleatoria normalmente distribuida de valor esperado $0$ y varianza $1$.

#### Caso multidimensional

El caso multidimensional es de la forma

$$dX_t = f(X_t,t)dt + g(X_t,t)dW_t\;\;\;\; (4)$$

donde $X_t,f(X_t,t) \in \mathbb{R}^d$, $g(X_t,t)\in \mathbb{R}^{d\times m}$ y $W_t\in \mathbb{R}^m$ para $d,m \in \mathbb{N}$.

En componentes

$$dX_{it} = f_i(X_t,t)dt + \sum_{j=1}^m g_{ij}(X_t,t)dW_{jt}\;\;\;\; (5)$$

Luego, la aproximación de Euler Maruyama toma la forma

$$Y_{i,t+\tau} = Y_{it} + f_i(Y_t,t)\tau + \sum_{j=1}^m g_{ij}(Y_t,t)w_j\sqrt{\tau}\mathcal{N}_j\;\;\;\; (6)$$

donde $\mathcal{N}_1,...,\mathcal{N}_m$ son variables aleatorias independientes, normalmente distribuidas, de valor esperado 0 y varianza 1.

### Red neuronal de Izhikevich

El modelo de neurona de Izhikevich es una Ecuación Diferencial Ordinaria (ODE), por lo que no hay estocasticidad.

Para simular una red, Izhikevich propone acoplar un sistema de neuronas con señales de ruido, tornando la ODE en una SDE.
Concretamente, propone acoplar $n_e$ neuronas exitatorias y $n_i$ neuronas inhibitorias mediante una matriz de interacciones $a_{ij}$ con $i,j\in \{1,2,...,n\}$ para $n:=n_e+n_i$ utilizando la ODE

\begin{eqnarray}
\dot{v}_i(t) & = &  g_2v_i^2(t) + g_1v_i(t) + g_0 - u_i(t) + \sum_{j=1}^n a_{ij} \Theta(v_j(t)) \;\;\;\; (7) \\
\dot{u}_i(t) & = &  b_i(c_iv_i(t)-u_i(t))
\end{eqnarray}

donde $I_i(t) = \sum_{j=1}^n a_{ij} \Theta(v_j(t))$ representa el input externo afectando a la neurona $i$.
Considerando valores $r_{ij}$ tomados aleatoria e independientemente de la distribución uniforme en $[0,1]$, se utiliza $a_{ij}=\frac{1}{2}r_{ij}$ para todo $i$ y $j=1,...,n_e$ (i.e. señales provenientes de neuronas exitatorias) y $a_{ij}=-r_{ij}$ para todo $i$ y $j=n_e+1,...,n$ (i.e. señales provenientes de neuronas inhibitorias).
Aquí, la función escalón de Heaviside, la cual satisface $\Theta(x)=1$ si $x>0$ y $\Theta(x)=0$ en caso contrario, es utilizada para normalizar la señal de disparo de las neuronas de entrada.
El disparo de la $i$-ésima neurona se implementa aplicando la condición de reseteo

\begin{eqnarray}
v_i(t) &\leftarrow& v_{i-} \;\;\;\; (8) \\
u_i(t) &\leftarrow& u_i(t) + \Delta u_i \\
\end{eqnarray}

cuando $v_i(t) \geq v_{i+}$.
Los valores de los parámetros para las variables exitatorias son son $b_i=0.02$, $c_i=0.2$, $v_{i-} = -64+15r_i^2$ y $\Delta u_i = 8-6r_i^2$, donde $R_i$ es la realización de una variable aleatoria definida uniformemente en el intervalo $[0,1]$.
Mientras que para las variables inhibitorias son $b_i=0.02+0.08r_i$, $c_i=0.25-0.05r_i$, $v_{i-} = -65$ y $\Delta u_i = 2$.
Esto introduce heterogeneida en el sistema, y la idea es que los límites $r_i=0$ corresponde neuronas que disparan regularmente, mientras que $r_i=1$ a neuronas *parloteantes*.

Luego, Izhikevich incorpora un ruido estocástico $dW_i(t)$ a la señal de entrada $I_i(t)$, de modo que el sistema de ODEs se transforma en un sistema de SDEs en donde las variables $v_i(t)$ y $u_i(t)$ se convierten en variables estocásticas $V_{i}(t)$ y $U_{i}(t)$, respectivamente.
De esta manera, en la aproximación de Euler-Maruyama, la Ec. (7) adopta la forma

\begin{eqnarray}
dV_{it} & = &  \bigg(g_2V_{it}^2 + g_1V_{it} + g_0 - U_{it} + \sum_{j=1}^n a_{ij} \Theta(V_{it})\bigg)dt + dK_{it} \;\;\;\; (9) \\
dU_{it} & = &  b_i(c_iV_{it}-U_{it})dt
\end{eqnarray}

donde $dK_{it}$ representa el ruido estocástico Gaussiano $dW_i(t)$ anteriormente mencionado.
Los valores $dK_{it}$ se obtienen de la distribución Gaussiana de media 0 y desviación estandard igual a $5$ para $i=1,...,n_e$ e igual a $2$ para $i=n_e+1,...,n$.
Obviando el reseteo de la Ec. (8), el sistema de SDEs de la Ec. (9) corresponde al sistema de la Ec. (5) cuando:

1. $X_{it} = V_{it}$,

2. $X_{i+n,t} = U_{it}$,

3. $f_i(X_t,t) = g_2X_{it}^2 + g_1X_{it} + g_0 - X_{i+n,t} + \sum_{j=1}^n a_{ij} \Theta(X_{it})$,

4. $f_{i+n}(X_t,t) = b_i(c_iX_{it}-X_{i+n,t})$,

para todo $i=1,...,n$,

5. y que

    \begin{eqnarray}
    g_{ij}(X_t,t)
    &=&
    \left\{
    \begin{array}{ll}
    5 & i=j\leq n_e, \\
    2 & n_e \leq i=j\leq n, \\
    0 & c.c.
    \end{array}
    \right.
    \end{eqnarray}

    de manera que

    $$
    dK_{it} = \sum_{j=1}^n g_{ij}(X_t,t) dW_{jt}
    $$

    donde $dW_{jt}$ es una variable aleatoria tomada de la distribución Gaussiana de media 0 y varianza 1 para todo tiempo $t$ y neurona $j=1,...,n$.

### Refs.

* https://en.wikipedia.org/wiki/Euler%E2%80%93Maruyama_method
* https://ipython-books.github.io/134-simulating-a-stochastic-differential-equation/

* https://github.com/mattja/sdeint/blob/master/sdeint/integrate.py

* https://diffeq.sciml.ai/stable/tutorials/sde_example/

**1)** Adapte el código en Matlab del paper de Izhikevich a código en Python.

**2)** Acomode el código anterior a la teoría mencionada remplazando la función escalón $\Theta$ de Heavised por la función

$$
z(v)=\frac{87+v}{450}-0.0193
$$

**3)** Reproduzca la figura 3 del paper. Sugerencia, compartimentabilice el código en funciones, identificando las distintas componentes mencionadas en la teoría.

In [ ]:
# 3.1) Adaptación del código Matlab de Izhikevich (2003) a Python usando Euler-Maruyama

# Fijamos semilla aleatoria para garantizar reproducibilidad
np.random.seed(42)

# Parámetros estructurales de la red
ne = 800  # Neuronas excitatorias
ni = 200  # Neuronas inhibitorias
n = ne + ni

# Realizaciones independientes uniformes en [0, 1]
r_e = np.random.rand(ne)
r_i = np.random.rand(ni)

# Parámetros heterogéneos de las neuronas (notación de la guía):
# b_i: escala de tiempo de la variable de recuperación u
b_vec = np.concatenate([0.02 * np.ones(ne), 0.02 + 0.08 * r_i])

# c_i: sensibilidad de u al voltaje subumbral v
c_vec = np.concatenate([0.2 * np.ones(ne), 0.25 - 0.05 * r_i])

# v_minus: potencial de reseteo tras el disparo
v_minus_vec = np.concatenate([-64.0 + 15.0 * (r_e**2), -65.0 * np.ones(ni)])

# delta_u: incremento de u tras un disparo
delta_u_vec = np.concatenate([8.0 - 6.0 * (r_e**2), 2.0 * np.ones(ni)])

# Umbral de disparo v+
v_plus = 30.0

# Parámetros fijos de la dinámica cuadrática
g2 = 0.04
g1 = 5.0
g0 = 140.0

# Matriz sináptica de interacciones a_ij (forma 1000 x 1000)
# Para j excitatorias (columnas 0 a ne-1): a_ij = 0.5 * rand()
# Para j inhibitorias (columnas ne a n-1): a_ij = -1.0 * rand()
a_mat = np.zeros((n, n))
a_mat[:, :ne] = 0.5 * np.random.rand(n, ne)
a_mat[:, ne:] = -1.0 * np.random.rand(n, ni)

# Desviación estándar del ruido Gaussiano talámico dK_i ~ N(0, sigma_i^2)
sigma_vec = np.concatenate([5.0 * np.ones(ne), 2.0 * np.ones(ni)])

# Condiciones iniciales estándar (neuronas en reposo)
v_init = -65.0 * np.ones(n)
u_init = c_vec * v_init

print("Red neuronal de Izhikevich configurada exitosamente:")
print(f"- Población total: {n} ({ne} excitatorias + {ni} inhibitorias)")
print(f"- Matriz sináptica A: {a_mat.shape}, rango de pesos [{a_mat.min():.2f}, {a_mat.max():.2f}]")
print(f"- Ruido estocástico talámico: sigma_exc = 5.0, sigma_inh = 2.0")

### **3.2) Justificación de la función de activación sináptica $z(v)$**

En el modelo computacional original de Izhikevich (2003), la transmisión sináptica es todo-o-nada y ocurre únicamente en el instante discreto en que una neurona presináptica supera el umbral de disparo ($v_j \ge 30$ mV), lo cual equivale a evaluar una función escalón $\Theta(v_j - 30)$.

En la formulación matemática continua del sistema de SDEs acopladas (Ecs. 7 y 9), la señal sináptica presináptica se representa mediante una función $z(v)$:

$$z(v) = \frac{87 + v}{450} - 0.0193$$

* **Comportamiento en reposo ($v = -65$ mV):**
  $$z(-65) = \frac{87 - 65}{450} - 0.0193 = \frac{22}{450} - 0.0193 \approx 0.0489 - 0.0193 = 0.0296$$
  En el reposo, $z(v)$ aporta una leve corriente sináptica basal tónica que, integrada sobre las 800 conexiones excitatorias y 200 inhibitorias, mantiene a la red en un estado despolarizado fisiológico.

* **Comportamiento en el pico del potencial de acción ($v = +30$ mV):**
  $$z(30) = \frac{87 + 30}{450} - 0.0193 = \frac{117}{450} - 0.0193 = 0.2600 - 0.0193 = 0.2407$$
  Durante un disparo, la activación sináptica se multiplica por un factor mayor a 8 ($0.2407 / 0.0296 \approx 8.13$), emulando la liberación masiva de neurotransmisores en la hendidura sináptica.

* **Punto nulo ($z(v) = 0$):** Ocurre a $v = -78.3$ mV. Por debajo de este potencial hiperpolarizado, la liberación es nula. Para garantizar estricta positividad física en simulaciones numéricas, puede emplearse la versión rectificada $\max(0, z(v))$.

In [ ]:
# 3.2) Implementación de la función sináptica z(v) y su comparación gráfica con Theta(v)

def z(v, rectificar=False):
    """
    Función de activación sináptica continua:
        z(v) = (87 + v) / 450 - 0.0193
        
    Argumentos:
        v : array con potenciales de membrana [mV]
        rectificar : bool, si True impone cota inferior 0 (ReLU biológico)
    """
    val = (87.0 + v) / 450.0 - 0.0193
    if rectificar:
        return np.maximum(0.0, val)
    return val


def theta_heaviside(v, v_th=0.0):
    """
    Función escalón de Heaviside:
        Theta(v) = 1 si v > v_th, 0 en caso contrario.
    """
    return np.where(v > v_th, 1.0, 0.0)


# Curva de comparación entre z(v) y Heaviside Theta(v)
v_rango = np.linspace(-95, 40, 500)
z_lineal = z(v_rango, rectificar=False)
z_rectificada = z(v_rango, rectificar=True)
th_curva = theta_heaviside(v_rango, v_th=0.0)

plt.figure(figsize=(9, 4.5))
plt.plot(v_rango, z_lineal, color='navy', lw=2, label='$z(v) = \frac{87+v}{450} - 0.0193$ (Lineal)')
plt.plot(v_rango, z_rectificada, color='royalblue', ls='--', lw=1.8, label='$z(v)$ rectificada $(\ge 0)$')
plt.plot(v_rango, th_curva, color='crimson', ls=':', lw=2, label='$\Theta(v)$ (Heaviside con umbral $0$ mV)')
plt.axvline(-65, color='gray', ls='--', lw=1, alpha=0.7, label='Reposo ($v = -65$ mV)')
plt.axvline(30, color='darkorange', ls='--', lw=1, alpha=0.7, label='Pico de espiga ($v = +30$ mV)')
plt.title('Función de activación sináptica: $z(v)$ vs $\Theta(v)$', fontsize=12, fontweight='bold')
plt.xlabel('Potencial de membrana $v$ [mV]', fontsize=11)
plt.ylabel('Señal sináptica de salida', fontsize=11)
plt.legend(framealpha=0.9, fontsize=9.5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 3.3) Simulación de la red con Euler-Maruyama y reproducción de la Figura 3 del paper

def simular_red_estocastica_izhikevich(T=1000, dt=1.0, modelo_sinapsis='z', seed=42):
    """
    Simula la red de 1000 neuronas estocásticas de Izhikevich usando Euler-Maruyama.
    
    Argumentos:
        T : tiempo total de integración [ms]
        dt : paso de tiempo [ms] (dt = 1.0 ms con dos medio-pasos de 0.5 ms para v)
        modelo_sinapsis : 'z' para z(v) continua, 'heaviside' para disparos discretos
        seed : semilla del generador de números aleatorios
        
    Retorna:
        firings_arr : array (M, 2) con [tiempo_ms, id_neurona] para cada disparo
        v_trace_0   : traza del potencial v(t) de la neurona excitatoria #0
        t_vec       : array temporal [0, 1, ..., T-1]
    """
    np.random.seed(seed)
    
    ne = 800
    ni = 200
    n = ne + ni
    
    r_e = np.random.rand(ne)
    r_i = np.random.rand(ni)
    
    b_v = np.concatenate([0.02 * np.ones(ne), 0.02 + 0.08 * r_i])
    c_v = np.concatenate([0.2 * np.ones(ne), 0.25 - 0.05 * r_i])
    v_min_v = np.concatenate([-64.0 + 15.0 * (r_e**2), -65.0 * np.ones(ni)])
    delta_u_v = np.concatenate([8.0 - 6.0 * (r_e**2), 2.0 * np.ones(ni)])
    
    a_m = np.zeros((n, n))
    a_m[:, :ne] = 0.5 * np.random.rand(n, ne)
    a_m[:, ne:] = -1.0 * np.random.rand(n, ni)
    
    sigma_v = np.concatenate([5.0 * np.ones(ne), 2.0 * np.ones(ni)])
    
    v = -65.0 * np.ones(n)
    u = c_v * v
    
    firings = []
    v_trace_0 = np.zeros(T)
    t_vec = np.arange(T)
    
    sqrt_dt = np.sqrt(dt)
    
    for t in range(T):
        # 1. Ruido estocástico talámico según aproximación de Euler-Maruyama:
        # dK_i = sigma_i * sqrt(dt) * N(0, 1)
        dK = sigma_v * sqrt_dt * np.random.randn(n)
        
        # 2. Detección de espigas y reseteo discontinuo
        fired = np.where(v >= 30.0)[0]
        if len(fired) > 0:
            for f_id in fired:
                firings.append((t, f_id))
            v[fired] = v_min_v[fired]
            u[fired] = u[fired] + delta_u_v[fired]
            
        # 3. Corriente sináptica recibida según el modelo
        if modelo_sinapsis == 'z':
            # Acoplamiento continuo z(v)
            s_in = z(v, rectificar=True)
            I_syn = a_m @ s_in
        else:
            # Acoplamiento Heaviside discreto original de Izhikevich
            if len(fired) > 0:
                I_syn = a_m[:, fired].sum(axis=1)
            else:
                I_syn = np.zeros(n)
                
        # Corriente total de entrada
        I_tot = I_syn + dK
        
        # 4. Integración estocástica de Euler-Maruyama:
        # Se utilizan dos pasos de 0.5*dt para el potencial v para asegurar estabilidad numérica
        # frente a la no linealidad cuadrática 0.04*v^2
        v = v + 0.5 * dt * (0.04 * (v**2) + 5.0 * v + 140.0 - u + I_tot)
        v = v + 0.5 * dt * (0.04 * (v**2) + 5.0 * v + 140.0 - u + I_tot)
        u = u + dt * b_v * (c_v * v - u)
        
        # Registramos el potencial de la neurona #0 para el panel inferior
        v_trace_0[t] = v[0]
        
    firings_arr = np.array(firings) if len(firings) > 0 else np.empty((0, 2))
    return firings_arr, v_trace_0, t_vec


# Ejecución de la simulación por T = 1000 ms con el modelo de activación z(v)
T_total = 1000
firings_data, v_trace_0, t_vec = simular_red_estocastica_izhikevich(T=T_total, dt=1.0, modelo_sinapsis='z', seed=42)

# Gráfico compuesto que reproduce la Figura 3 del paper de Izhikevich
fig, (ax_raster, ax_volt) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, 
                                         gridspec_kw={'height_ratios': [2.4, 1.2]})

# Panel Superior: Raster Plot de las 1000 neuronas
if len(firings_data) > 0:
    t_spk = firings_data[:, 0]
    id_spk = firings_data[:, 1]
    
    mask_e = id_spk < 800
    mask_i = id_spk >= 800
    
    ax_raster.scatter(t_spk[mask_e], id_spk[mask_e], s=1.2, color='navy', alpha=0.75, label='Excitatorias (0-799)')
    ax_raster.scatter(t_spk[mask_i], id_spk[mask_i], s=1.2, color='crimson', alpha=0.85, label='Inhibitorias (800-999)')

ax_raster.axhline(800, color='black', ls='--', lw=1, alpha=0.6)
ax_raster.set_ylabel('Índice de neurona', fontsize=12)
ax_raster.set_ylim(-10, 1010)
ax_raster.set_title('Figura 3: Dinámica colectiva y raster plot de la red de 1000 neuronas de Izhikevich', fontsize=13, fontweight='bold')
ax_raster.legend(loc='upper right', framealpha=0.9, markerscale=3.5)
ax_raster.grid(True, alpha=0.25)

# Panel Inferior: Voltaje de una neurona excitatoria representativa (índice 0)
ax_volt.plot(t_vec, v_trace_0, color='royalblue', lw=1.0, label='Neurona excitatoria #0')
ax_volt.axhline(30.0, color='crimson', ls=':', lw=0.9, alpha=0.7, label='$v_+ = 30$ mV (Umbral)')
ax_volt.set_xlabel('Tiempo $t$ [ms]', fontsize=12)
ax_volt.set_ylabel('$v$ [mV]', fontsize=12)
ax_volt.set_xlim(0, T_total)
ax_volt.set_ylim(-90, 45)
ax_volt.legend(loc='upper right', framealpha=0.9)
ax_volt.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

### **Conclusiones y discusión científica**

1. **Riqueza dinámica del modelo de Izhikevich:**
   A diferencia de los modelos tradicionales *Integrate-and-Fire* lineales, la no linealidad cuadrática $g_2 v^2 + g_1 v + g_0$ combinada con la variable de recuperación $u(t)$ y el mecanismo discontinuo de reseteo permite reproducir fielmente la amplia zoología electrofisiológica observada in vivo (los 8 paneles de la Figura 2), manteniendo un costo computacional mínimo comparable al de un modelo lineal de una dimensión.

2. **Aproximación de Euler-Maruyama en redes estocásticas:**
   La inclusión de ruido Gaussiano talámico mediante el esquema de Euler-Maruyama ($dK_i = \sigma_i \sqrt{\Delta t} \mathcal{N}(0, 1)$) junto con el escalado heterogéneo de parámetros modela de forma realista la variabilidad neuronal intrínseca y la entrada sináptica no correlacionada de fondo.

3. **Transmisión sináptica continua $z(v)$:**
   Reemplazar la función escalón idealizada $\Theta(v)$ por la función de activación continua $z(v) = \frac{87+v}{450} - 0.0193$ proporciona una formulación analíticamente tratable del acoplamiento sináptico, reflejando tanto la despolarización tónica en reposo como el incremento abrupto de transmisión durante el disparo de potenciales de acción.

4. **Aparición espontánea de ritmos colectivos (Figura 3):**
   En la red de 1000 neuronas, el balance dinámico entre la excitación recurrente ($80\%$) y la inhibición por retroalimentación ($20\%$) genera espontáneamente una sincronización parcial con oscilaciones colectivas en la banda gamma (alrededor de 40 Hz), reproduciendo los patrones electroencefalográficos y de potenciales de campo local (LFP) característicos de la corteza de mamíferos.